# VidRec-Lens: content-aware video recommendation with bias-aware evaluation

Run top to bottom (Run All). Before starting, in the right-hand panel under Session options set **Accelerator: GPU T4** and **Internet: On**.
Cells are safe to re-run: downloads resume, finished steps are skipped.
Set the flags in the next cell to choose which model steps run (the defaults skip the long training steps and run the bias analysis).

In [1]:
RUN_BASELINES = False
RUN_TWOTOWER = False
RUN_REFIT = False
RUN_BIAS = True
RUN_AB = True
RUN_CLUSTERS = True
LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RUN_EXPORT = True


In [2]:
!nvidia-smi -L
!df -h /kaggle/working

GPU 0: Tesla T4 (UUID: GPU-5bf90663-f398-f0ee-b0a5-c49385fec90d)
GPU 1: Tesla T4 (UUID: GPU-6d23961c-3ef5-ff93-baad-b595e7b2665c)
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G   64K   20G   1% /kaggle/working


## 1. Download MicroLens-100K files (resumable)

In [3]:
%%bash
mkdir -p /kaggle/working/data && cd /kaggle/working/data
B=https://recsys.westlake.edu.cn/MicroLens-100k-Dataset
for f in readme.txt MicroLens-100k_title_en.csv MicroLens-100k_covers.zip extracted_modality_features.zip; do
  wget -q -c --show-progress "$B/$f"
done
ls -la

total 1371404
drwxr-xr-x 2 root root      4096 Oct 25 12:55 .
drwxr-xr-x 3 root root      4096 Oct 25 12:51 ..
-rw-r--r-- 1 root root 717084588 Mar  3  2026 extracted_modality_features.zip
-rw-r--r-- 1 root root 685706907 Mar  3  2026 MicroLens-100k_covers.zip
-rw-r--r-- 1 root root   1500488 Mar  3  2026 MicroLens-100k_title_en.csv
-rw-r--r-- 1 root root       615 Mar  3  2026 readme.txt



     0K                                                       100%  194M=0s


     0K .......... .......... .......... .......... ..........  3%  215K 7s
    50K .......... .......... .......... .......... ..........  6%  202K 7s
   100K .......... .......... .......... .......... .......... 10%  193K 6s
   150K .......... .......... .......... .......... .......... 13% 71.3M 5s
   200K .......... .......... .......... .......... .......... 17%  188K 5s
   250K .......... .......... .......... .......... .......... 20%  105M 4s
   300K .......... .......... .......... .......... .......... 23% 52.9M 3s
   350K .......... .......... .......... .......... .......... 27%  186K 3s
   400K .......... .......... .......... .......... .......... 30% 69.8M 3s
   450K .......... .......... .......... .......... .......... 34%  190K 3s
   500K .......... .......... .......... .......... .......... 37% 55.6M 3s
   550K .......... .......... .......... .......... .......... 40% 35.3M 2s
   600K .

In [4]:
%%bash
cd /kaggle/working/data
if [ -d covers ] && [ "$(ls covers | wc -l)" -ge 19738 ]; then echo "covers already unzipped"; else rm -rf covers; unzip -oq MicroLens-100k_covers.zip -d covers < /dev/null; fi
if [ -f feats/MicroLens-100k_video_features_VideoMAE.npy ]; then echo "feats already unzipped"; else rm -rf feats; unzip -oq extracted_modality_features.zip -d feats < /dev/null; fi
find covers -type f | wc -l
ls feats

19738
MicroLens-100k_image_features_CLIPRN50.json
MicroLens-100k_image_features_CLIPRN50.npy
MicroLens-100k_title_en_text_features_BgeM3.json
MicroLens-100k_title_en_text_features_BgeM3.npy
MicroLens-100k_video_features_VideoMAE.json
MicroLens-100k_video_features_VideoMAE.npy


## 2. Embeddings: CLIP ViT-B/32 on covers (ours) and the authors' VideoMAE, CLIP RN50 and BGE-M3 features

In [5]:
import os, numpy as np, torch

if os.path.exists("/kaggle/working/emb/clip_vitb32_covers.npy"):
    print("CLIP cover embeddings already exist, skipping")
else:
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader
    from transformers import CLIPModel, CLIPProcessor

    D = "/kaggle/working/data/covers"
    ids = sorted(int(f[:-4]) for f in os.listdir(D) if f.endswith(".jpg"))
    proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32", torch_dtype=torch.float16).cuda().eval()

    class DS(Dataset):
        def __len__(self):
            return len(ids)

        def __getitem__(self, i):
            im = Image.open(f"{D}/{ids[i]}.jpg").convert("RGB")
            return proc(images=im, return_tensors="pt")["pixel_values"][0]

    dl = DataLoader(DS(), batch_size=256, num_workers=4)
    out = []
    with torch.inference_mode():
        for x in dl:
            f = model.get_image_features(pixel_values=x.cuda().half())
            f = f if torch.is_tensor(f) else f.pooler_output
            out.append(torch.nn.functional.normalize(f.float(), dim=-1).cpu())

    emb = torch.cat(out).numpy().astype(np.float16)
    os.makedirs("/kaggle/working/emb", exist_ok=True)
    np.save("/kaggle/working/emb/clip_vitb32_covers.npy", emb)
    np.save("/kaggle/working/emb/clip_vitb32_covers_ids.npy", np.array(ids))
    print(emb.shape)

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

(19738, 512)


In [6]:
import os, json, numpy as np

P = "/kaggle/working/data/feats/"
os.makedirs("/kaggle/working/emb", exist_ok=True)
names = {
    "videomae": "MicroLens-100k_video_features_VideoMAE",
    "cliprn50": "MicroLens-100k_image_features_CLIPRN50",
    "bgem3": "MicroLens-100k_title_en_text_features_BgeM3",
}
for k, n in names.items():
    a = np.load(P + n + ".npy", mmap_mode="r")
    j = json.load(open(P + n + ".json"))
    keys = list(j.keys())
    ids = np.array([int(x) for x in keys])
    v = np.array([j[x] for x in keys], dtype=np.float32)
    print(k, a.shape, v.shape, "npy matches json:", np.allclose(a, v, atol=1e-4), "ids sorted:", bool((np.diff(ids) > 0).all()))
    np.save(f"/kaggle/working/emb/{k}.npy", v.astype(np.float16))
    np.save(f"/kaggle/working/emb/{k}_ids.npy", ids)
print(sorted(os.listdir("/kaggle/working/emb")))

videomae (19738, 768) (19738, 768) npy matches json: True ids sorted: True
cliprn50 (19738, 1024) (19738, 1024) npy matches json: True ids sorted: True
bgem3 (19738, 1024) (19738, 1024) npy matches json: True ids sorted: True
['bgem3.npy', 'bgem3_ids.npy', 'clip_vitb32_covers.npy', 'clip_vitb32_covers_ids.npy', 'cliprn50.npy', 'cliprn50_ids.npy', 'videomae.npy', 'videomae_ids.npy']


## 3. Write the project source (single source of truth for the pipeline)

In [7]:
%%bash
ROOT=/kaggle/working/vidreclens
mkdir -p $ROOT/src/vidreclens/data $ROOT/src/vidreclens/features $ROOT/src/vidreclens/models $ROOT/src/vidreclens/eval $ROOT/src/vidreclens/serve $ROOT/scripts $ROOT/tests $ROOT/data/raw $ROOT/data/processed $ROOT/reports
for d in '' data features models eval serve; do touch $ROOT/src/vidreclens/$d/__init__.py; done
cat > $ROOT/pyproject.toml <<'EOF'
[build-system]
requires = ["setuptools"]
build-backend = "setuptools.build_meta"

[project]
name = "vidreclens"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = ["polars>=1.0", "numpy", "matplotlib", "pyarrow"]

[project.optional-dependencies]
dev = ["pytest", "httpx"]
serve = ["fastapi", "uvicorn"]

[tool.setuptools.packages.find]
where = ["src"]
EOF
cat > $ROOT/pytest.ini <<'EOF'
[pytest]
pythonpath = src
testpaths = tests
EOF
cat > $ROOT/.gitignore <<'EOF'
data/raw/*
data/processed/*
!data/raw/.gitkeep
!data/processed/.gitkeep
__pycache__/
*.egg-info/
.venv/
EOF
cat > $ROOT/.dockerignore <<'EOF'
data/
reports/
tests/
scripts/
.venv/
__pycache__/
*.pt
*.egg-info/
EOF
cat > $ROOT/Dockerfile <<'EOF'
FROM python:3.11-slim
WORKDIR /app
COPY requirements-serve.txt .
RUN pip install --no-cache-dir -r requirements-serve.txt
COPY src/vidreclens/__init__.py ./vidreclens/__init__.py
COPY src/vidreclens/serve ./vidreclens/serve
COPY bundle ./bundle
ENV PYTHONPATH=/app
ENV VIDRECLENS_BUNDLE=/app/bundle
EXPOSE 8000
CMD ["uvicorn", "--factory", "vidreclens.serve.app:create_app", "--host", "0.0.0.0", "--port", "8000"]
EOF
cat > $ROOT/requirements-serve.txt <<'EOF'
fastapi>=0.110
uvicorn>=0.29
numpy>=1.26
EOF
cat > $ROOT/src/vidreclens/config.py <<'EOF'
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5
EOF
cat > $ROOT/src/vidreclens/data/load.py <<'EOF'
from pathlib import Path

import polars as pl

from vidreclens.config import RAW


def find_pairs_file(root: Path = RAW) -> Path:
    hits = sorted(Path(root).rglob("*pairs.csv"))
    if not hits:
        raise FileNotFoundError(f"no *pairs.csv under {root}")
    return hits[0]


def load_interactions(root: Path = RAW, path: Path | None = None) -> pl.LazyFrame:
    lf = pl.scan_csv(path or find_pairs_file(root))
    cols = lf.collect_schema().names()[:3]
    lf = lf.select(
        pl.col(cols[0]).alias("user"),
        pl.col(cols[1]).alias("item"),
        pl.col(cols[2]).alias("ts"),
    )
    dtype = lf.collect_schema()["ts"]
    raw = pl.col("ts")
    if dtype == pl.String:
        raw = pl.coalesce(
            raw.cast(pl.Int64, strict=False),
            raw.str.to_datetime(strict=False).dt.epoch("s"),
        )
    elif isinstance(dtype, pl.Datetime):
        raw = raw.dt.epoch("s")
    ts = pl.when(raw > 10**11).then(raw // 1000).otherwise(raw)
    return lf.select(
        pl.col("user").cast(pl.Int64, strict=False),
        pl.col("item").cast(pl.Int64, strict=False),
        ts.cast(pl.Int64).alias("ts"),
    )


def load_item_stats(root: Path = RAW, path: Path | None = None) -> pl.DataFrame:
    path = path or next(Path(root).rglob("*likes_and_views.txt"))
    raw = pl.read_csv(
        path, has_header=False, separator="\x1f", new_columns=["line"], infer_schema=False
    )
    g = pl.col("line").str.extract_groups(r"^\s*(\d+)\s+(\d+)\s+(\d+)").alias("g")
    return (
        raw.select(g)
        .unnest("g")
        .rename({"1": "item", "2": "likes", "3": "views"})
        .drop_nulls()
        .with_columns(pl.all().cast(pl.Int64))
        .unique(subset="item", keep="first")
    )
EOF
cat > $ROOT/src/vidreclens/data/clean.py <<'EOF'
import polars as pl


def kcore(df: pl.DataFrame, min_user: int, min_item: int) -> pl.DataFrame:
    while True:
        n = df.height
        df = df.filter(pl.len().over("item") >= min_item)
        df = df.filter(pl.len().over("user") >= min_user)
        if df.height == n:
            return df


def reindex(df: pl.DataFrame):
    users = df.select("user").unique().sort("user").with_row_index("uid")
    items = df.select("item").unique().sort("item").with_row_index("iid")
    users = users.with_columns(pl.col("uid").cast(pl.Int64))
    items = items.with_columns(pl.col("iid").cast(pl.Int64))
    out = df.join(users, on="user").join(items, on="item").select("uid", "iid", "ts")
    return out.sort("uid", "ts", "iid"), users, items


def clean_interactions(lf: pl.LazyFrame, min_user: int, min_item: int):
    df = (
        lf.drop_nulls()
        .sort("ts", "user", "item")
        .unique(subset=["user", "item"], keep="first", maintain_order=True)
        .collect()
    )
    return reindex(kcore(df, min_user, min_item))
EOF
cat > $ROOT/src/vidreclens/data/split.py <<'EOF'
import polars as pl


def leave_last_out(df: pl.DataFrame) -> pl.DataFrame:
    df = df.sort("uid", "ts", "iid").with_columns(
        pl.int_range(pl.len()).over("uid").alias("pos"),
        pl.len().over("uid").alias("n"),
    )
    split = (
        pl.when(pl.col("pos") == pl.col("n") - 1)
        .then(pl.lit("test"))
        .when(pl.col("pos") == pl.col("n") - 2)
        .then(pl.lit("val"))
        .otherwise(pl.lit("train"))
    )
    return df.with_columns(split.alias("split")).drop("pos", "n")
EOF
cat > $ROOT/src/vidreclens/data/eda.py <<'EOF'
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import polars as pl


def gini(x: np.ndarray) -> float:
    x = np.sort(x.astype(np.float64))
    c = np.cumsum(x)
    return float((len(x) + 1 - 2 * c.sum() / c[-1]) / len(x))


def item_counts(df: pl.DataFrame) -> np.ndarray:
    return df.group_by("iid").len()["len"].to_numpy()


def summarize(df: pl.DataFrame) -> dict:
    ic = item_counts(df)
    uc = df.group_by("uid").len()["len"].to_numpy()
    srt = np.sort(ic)[::-1]
    k = max(1, int(0.2 * len(srt)))
    n_users, n_items, n = len(uc), len(ic), df.height
    return {
        "users": n_users,
        "items": n_items,
        "interactions": n,
        "sparsity": 1 - n / (n_users * n_items),
        "seq_len_p50": float(np.percentile(uc, 50)),
        "seq_len_p90": float(np.percentile(uc, 90)),
        "seq_len_max": int(uc.max()),
        "item_gini": gini(ic),
        "top20pct_items_share": float(srt[:k].sum() / srt.sum()),
        "span_days": (df["ts"].max() - df["ts"].min()) / 86400,
    }


def plot_popularity(df: pl.DataFrame, path: Path) -> None:
    srt = np.sort(item_counts(df))[::-1]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.loglog(np.arange(1, len(srt) + 1), srt)
    ax.set_xlabel("item rank")
    ax.set_ylabel("interactions")
    ax.set_title("Item popularity (long tail)")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
EOF
cat > $ROOT/src/vidreclens/data/history.py <<'EOF'
import numpy as np
import polars as pl


def padded_history(df: pl.DataFrame, splits, n_users: int, n_items: int, max_len: int) -> np.ndarray:
    d = df.filter(pl.col("split").is_in(list(splits))).sort("uid", "ts", "iid")
    uid = d["uid"].to_numpy()
    iid = d["iid"].to_numpy()
    counts = np.bincount(uid, minlength=n_users)
    start = np.concatenate([[0], np.cumsum(counts)[:-1]])
    pos = np.arange(len(uid)) - start[uid]
    drop = np.maximum(counts - max_len, 0)[uid]
    keep = pos >= drop
    P = np.full((n_users, max_len), n_items, dtype=np.int64)
    P[uid[keep], (pos - drop)[keep]] = iid[keep]
    return P
EOF
cat > $ROOT/src/vidreclens/eval/metrics.py <<'EOF'
import numpy as np

KS = (10, 50)


def tie_aware_rank(scores: np.ndarray, target: np.ndarray) -> np.ndarray:
    t = scores[np.arange(len(target)), target][:, None]
    greater = (scores > t).sum(1)
    equal = (scores == t).sum(1)
    return 1 + greater + (equal - 1) / 2


def summarize(ranks: np.ndarray, head: np.ndarray, ks=KS) -> dict:
    out = {}
    for k in ks:
        hit = ranks <= k
        out[f"recall@{k}"] = float(hit.mean())
        out[f"ndcg@{k}"] = float(np.where(hit, 1 / np.log2(ranks + 1), 0).mean())
        out[f"recall@{k}_head"] = float(hit[head].mean())
        out[f"recall@{k}_tail"] = float(hit[~head].mean())
    return out


def evaluate(score_fn, target: np.ndarray, head_items: np.ndarray, batch: int = 2048) -> dict:
    n = len(target)
    ranks = np.empty(n)
    for a in range(0, n, batch):
        idx = np.arange(a, min(a + batch, n))
        ranks[idx] = tie_aware_rank(score_fn(idx), target[idx])
    return summarize(ranks, head_items[target])
EOF
cat > $ROOT/src/vidreclens/features/io.py <<'EOF'
from pathlib import Path

import numpy as np


def load_features(feat_dir: Path, names) -> dict:
    out = {}
    for n in names:
        x = np.load(Path(feat_dir) / f"{n}.npy").astype(np.float32)
        x = (x - x.mean(0)) / (x.std(0) + 1e-6)
        out[n] = x / np.linalg.norm(x, axis=1, keepdims=True).clip(1e-8)
    return out
EOF
cat > $ROOT/src/vidreclens/models/baselines.py <<'EOF'
import numpy as np
import polars as pl
import scipy.sparse as sp


def history_matrix(df: pl.DataFrame, n_users: int, n_items: int, splits) -> sp.csr_matrix:
    d = df.filter(pl.col("split").is_in(list(splits)))
    data = np.ones(d.height, dtype=np.float32)
    return sp.csr_matrix((data, (d["uid"].to_numpy(), d["iid"].to_numpy())), shape=(n_users, n_items))


def mask_history(s: np.ndarray, H: sp.csr_matrix, idx: np.ndarray) -> np.ndarray:
    sub = H[idx]
    rows = np.repeat(np.arange(len(idx)), np.diff(sub.indptr))
    s[rows, sub.indices] = -np.inf
    return s


def popularity(H):
    pop = np.asarray(H.sum(0)).ravel().astype(np.float32)

    def fn(idx):
        return mask_history(np.tile(pop, (len(idx), 1)), H, idx)

    return fn


def content_profile(H, E):
    deg = np.asarray(H.sum(1)).ravel().clip(1)
    U = np.asarray(H @ E) / deg[:, None]
    U /= np.linalg.norm(U, axis=1, keepdims=True).clip(1e-8)

    def fn(idx):
        return mask_history(U[idx] @ E.T, H, idx)

    return fn


def itemknn(H):
    G = (H.T @ H).toarray().astype(np.float32)
    d = np.sqrt(np.diag(G)).clip(1e-8)
    G /= d[:, None]
    G /= d[None, :]
    np.fill_diagonal(G, 0)

    def fn(idx):
        return mask_history(np.asarray(H[idx] @ G), H, idx)

    return fn
EOF
cat > $ROOT/src/vidreclens/models/twotower.py <<'EOF'
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class TwoTower(nn.Module):
    def __init__(self, feats, n_items, d=128, use_id=True, id_drop=0.3, mod_drop=0.2, drop=0.1):
        super().__init__()
        self.names = list(feats)
        self.id_drop = id_drop
        self.mod_drop = mod_drop
        for k, v in feats.items():
            self.register_buffer(f"f_{k}", v, persistent=False)
        self.proj = nn.ModuleDict({k: nn.Linear(v.shape[1], d) for k, v in feats.items()})
        self.id = nn.Embedding(n_items, d) if use_id else None
        if self.id is not None:
            nn.init.normal_(self.id.weight, std=0.1)
        self.norm = nn.LayerNorm(d)
        self.item_mlp = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(drop), nn.Linear(d, d))
        self.user_mlp = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(drop), nn.Linear(d, d))
        self.log_scale = nn.Parameter(torch.tensor(math.log(14.0)))

    def encode_items(self):
        h = 0
        for k in self.names:
            z = self.proj[k](getattr(self, f"f_{k}"))
            if self.training and self.mod_drop > 0:
                z = z * (torch.rand(z.size(0), 1, device=z.device) > self.mod_drop)
            h = h + z
        if self.id is not None:
            e = self.id.weight
            if self.training and self.id_drop > 0:
                e = e * (torch.rand(e.size(0), 1, device=e.device) > self.id_drop)
            h = h + e
        out = F.normalize(self.item_mlp(self.norm(h)), dim=-1)
        return torch.cat([out, out.new_zeros(1, out.size(1))])

    def encode_users(self, IR, hist, mask):
        m = mask.unsqueeze(-1).to(IR.dtype)
        x = (IR[hist] * m).sum(1) / m.sum(1).clamp(min=1)
        return F.normalize(self.user_mlp(x), dim=-1)
EOF
cat > $ROOT/src/vidreclens/serve/engine.py <<'EOF'
import csv
import json
import math
from pathlib import Path

import numpy as np

_ERF = np.vectorize(math.erf, otypes=[np.float64])


class Recommender:
    def __init__(self, bundle_dir):
        d = Path(bundle_dir)
        z = np.load(d / "model.npz")
        meta = json.loads((d / "meta.json").read_text())
        self.V = z["item_vecs"].astype(np.float32)
        self.W1, self.b1, self.W2, self.b2 = (z[k].astype(np.float32) for k in ("W1", "b1", "W2", "b2"))
        self.logpop = z["logpop"].astype(np.float32)
        self.pop = z["pop_counts"]
        self.item_ids = z["item_ids"]
        self.index = {int(v): i for i, v in enumerate(self.item_ids)}
        self.scale = float(meta["scale"])
        self.alpha = float(meta["alpha"])
        self.max_len = int(meta["max_len"])
        with open(d / "items.csv", newline="", encoding="utf-8") as f:
            titles = {int(r["item"]): r["title"] for r in csv.DictReader(f)}
        self.titles = [titles.get(int(v), "") for v in self.item_ids]
        self.by_pop = np.argsort(-self.pop, kind="stable")

    def user_vector(self, idx):
        x = self.V[idx].mean(0)
        h = x @ self.W1.T + self.b1
        h = 0.5 * h * (1.0 + _ERF(h / math.sqrt(2.0)))
        u = (h @ self.W2.T + self.b2).astype(np.float32)
        return u / max(float(np.linalg.norm(u)), 1e-12)

    def item(self, item_id):
        i = self.index.get(int(item_id))
        if i is None:
            return None
        return {"item_id": int(item_id), "title": self.titles[i], "past_interactions": int(self.pop[i])}

    def recommend(self, history, k=10, alpha=None):
        a = self.alpha if alpha is None else float(alpha)
        known, unknown, seen = [], [], set()
        for h in history:
            if h in seen:
                continue
            seen.add(h)
            i = self.index.get(int(h))
            (known if i is not None else unknown).append(i if i is not None else int(h))
        known = known[-self.max_len :]
        n = len(self.item_ids)
        if not known:
            top = self.by_pop[:k]
            scores = self.pop[top].astype(float)
            cold = True
        else:
            u = self.user_vector(known)
            s = self.scale * (self.V @ u) - a * self.logpop
            s[known] = -np.inf
            kk = min(k, n - len(known))
            top = np.argpartition(-s, kk - 1)[:kk]
            top = top[np.argsort(-s[top])]
            scores = s[top]
            cold = False
        items = [{"item_id": int(self.item_ids[j]), "title": self.titles[j], "score": float(sc)} for j, sc in zip(top, scores)]
        return {"items": items, "unknown_items": unknown, "cold_start": cold, "alpha": a}
EOF
cat > $ROOT/src/vidreclens/serve/app.py <<'EOF'
import os
from contextlib import asynccontextmanager
from pathlib import Path

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from vidreclens.serve.engine import Recommender


class RecommendRequest(BaseModel):
    history: list[int] = Field(default_factory=list, max_length=500, description="video ids the user watched, oldest first")
    k: int = Field(10, ge=1, le=100)
    alpha: float | None = Field(None, ge=0.0, le=2.0, description="popularity penalty, omit to use the default")


def create_app(bundle_dir=None):
    path = Path(bundle_dir or os.environ.get("VIDRECLENS_BUNDLE", "bundle"))

    @asynccontextmanager
    async def lifespan(app):
        app.state.engine = Recommender(path)
        yield

    app = FastAPI(title="VidRec-Lens", version="0.1.0", lifespan=lifespan)

    @app.get("/health")
    def health():
        e = app.state.engine
        return {"status": "ok", "items": len(e.item_ids), "default_alpha": e.alpha}

    @app.post("/recommend")
    def recommend(req: RecommendRequest):
        return app.state.engine.recommend(req.history, req.k, req.alpha)

    @app.get("/items/{item_id}")
    def item(item_id: int):
        r = app.state.engine.item(item_id)
        if r is None:
            raise HTTPException(status_code=404, detail="unknown item")
        return r

    return app
EOF
cat > $ROOT/src/vidreclens/serve/export.py <<'EOF'
import csv
import json
from pathlib import Path

import numpy as np


def export_bundle(model, item_ids, titles, pop_counts, out_dir, alpha=0.75, max_len=50):
    import torch

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    model.eval()
    with torch.no_grad():
        IR = model.encode_items().cpu().numpy().astype(np.float32)
    lin1, lin2 = model.user_mlp[0], model.user_mlp[3]
    pop_counts = np.asarray(pop_counts, dtype=np.int64)
    np.savez_compressed(
        out / "model.npz",
        item_vecs=IR[:-1],
        W1=lin1.weight.detach().cpu().numpy(),
        b1=lin1.bias.detach().cpu().numpy(),
        W2=lin2.weight.detach().cpu().numpy(),
        b2=lin2.bias.detach().cpu().numpy(),
        logpop=np.log1p(pop_counts).astype(np.float32),
        pop_counts=pop_counts,
        item_ids=np.asarray(item_ids, dtype=np.int64),
    )
    meta = {"scale": float(min(model.log_scale.exp().item(), 100.0)), "alpha": float(alpha), "max_len": int(max_len), "dim": int(IR.shape[1]), "n_items": int(len(item_ids))}
    (out / "meta.json").write_text(json.dumps(meta, indent=2))
    with open(out / "items.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["item", "title"])
        for i, t in zip(item_ids, titles):
            w.writerow([int(i), t])
    return out
EOF
cat > $ROOT/scripts/day1_prepare.py <<'EOF'
import argparse
import json
from pathlib import Path

import polars as pl

from vidreclens.config import MIN_ITEM_INTERACTIONS, MIN_USER_INTERACTIONS, PROCESSED, RAW, REPORTS
from vidreclens.data.clean import clean_interactions
from vidreclens.data.eda import plot_popularity, summarize
from vidreclens.data.load import load_interactions
from vidreclens.data.split import leave_last_out


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--raw", type=Path, default=RAW)
    p.add_argument("--out", type=Path, default=PROCESSED)
    p.add_argument("--reports", type=Path, default=REPORTS)
    p.add_argument("--min-user", type=int, default=MIN_USER_INTERACTIONS)
    p.add_argument("--min-item", type=int, default=MIN_ITEM_INTERACTIONS)
    a = p.parse_args()
    a.out.mkdir(parents=True, exist_ok=True)
    a.reports.mkdir(parents=True, exist_ok=True)

    lf = load_interactions(a.raw)
    n_raw = lf.select(pl.len()).collect().item()
    df, users, items = clean_interactions(lf, a.min_user, a.min_item)
    df = leave_last_out(df)

    df.write_parquet(a.out / "interactions.parquet")
    users.write_parquet(a.out / "users.parquet")
    items.write_parquet(a.out / "items.parquet")

    stats = {"raw_rows": n_raw, **summarize(df)}
    (a.reports / "stats.json").write_text(json.dumps(stats, indent=2))
    plot_popularity(df, a.reports / "popularity_longtail.png")
    print(json.dumps(stats, indent=2))


if __name__ == "__main__":
    main()
EOF
cat > $ROOT/scripts/day1b_item_stats.py <<'EOF'
import argparse
from pathlib import Path

import polars as pl

from vidreclens.config import PROCESSED, RAW
from vidreclens.data.load import load_item_stats


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--raw", type=Path, default=RAW)
    p.add_argument("--out", type=Path, default=PROCESSED)
    a = p.parse_args()

    items = pl.read_parquet(a.out / "items.parquet")
    stats = (
        items.join(load_item_stats(a.raw), on="item", how="left")
        .with_columns(
            (pl.col("likes") / pl.col("views")).alias("like_rate"),
            pl.col("views").log1p().alias("log_views"),
        )
        .select("iid", "item", "likes", "views", "like_rate", "log_views")
    )
    stats.write_parquet(a.out / "item_stats.parquet")
    print(stats.describe())
    print("matched:", stats["views"].is_not_null().mean())


if __name__ == "__main__":
    main()
EOF
cat > $ROOT/scripts/day2_join_features.py <<'EOF'
import argparse
from pathlib import Path

import numpy as np
import polars as pl

from vidreclens.config import PROCESSED

NAMES = ["clip_vitb32_covers", "videomae", "cliprn50", "bgem3"]


def aligned(emb_dir: Path, name: str, items: np.ndarray) -> np.ndarray:
    v = np.load(emb_dir / f"{name}.npy").astype(np.float32)
    ids = np.load(emb_dir / f"{name}_ids.npy")
    order = np.argsort(ids)
    srt = ids[order]
    idx = np.clip(np.searchsorted(srt, items), 0, len(srt) - 1)
    missing = int((srt[idx] != items).sum())
    if missing:
        raise ValueError(f"{name}: {missing} items without features")
    return v[order[idx]]


def topk(x: np.ndarray, sample: np.ndarray, k: int) -> np.ndarray:
    x = x / np.linalg.norm(x, axis=1, keepdims=True).clip(1e-8)
    s = x[sample] @ x.T
    s[np.arange(len(sample)), sample] = -np.inf
    return np.argpartition(-s, k, axis=1)[:, :k]


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--emb", type=Path, default=PROCESSED.parent / "embeddings")
    p.add_argument("--processed", type=Path, default=PROCESSED)
    p.add_argument("--k", type=int, default=10)
    a = p.parse_args()

    items = pl.read_parquet(a.processed / "items.parquet").sort("iid")["item"].to_numpy()
    out = a.processed / "features"
    out.mkdir(exist_ok=True)

    feats = {}
    for n in NAMES:
        feats[n] = aligned(a.emb, n, items)
        np.save(out / f"{n}.npy", feats[n].astype(np.float16))
        print(n, feats[n].shape)

    rng = np.random.default_rng(0)
    sample = rng.choice(len(items), size=min(500, len(items)), replace=False)
    nn = {n: topk(f, sample, a.k) for n, f in feats.items()}
    print("neighbour overlap@%d between modalities" % a.k)
    for i, m in enumerate(NAMES):
        for n in NAMES[i + 1 :]:
            ov = np.mean([len(set(x) & set(y)) / a.k for x, y in zip(nn[m], nn[n])])
            print(f"  {m} vs {n}: {ov:.3f}")


if __name__ == "__main__":
    main()
EOF
cat > $ROOT/scripts/day3_baselines.py <<'EOF'
import argparse
import json
from pathlib import Path

import numpy as np
import polars as pl

from vidreclens.config import PROCESSED, REPORTS
from vidreclens.eval.metrics import evaluate
from vidreclens.features.io import load_features
from vidreclens.models.baselines import content_profile, history_matrix, itemknn, popularity

NAMES = ["clip_vitb32_covers", "videomae", "cliprn50", "bgem3"]
COLS = ["recall@10", "ndcg@10", "recall@50", "ndcg@50", "recall@10_head", "recall@10_tail"]


def show(rows):
    print(f"{'model':<26}" + "".join(f"{c:>16}" for c in COLS))
    for name, r in rows.items():
        print(f"{name:<26}" + "".join(f"{r[c]:>16.4f}" for c in COLS))


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--processed", type=Path, default=PROCESSED)
    p.add_argument("--reports", type=Path, default=REPORTS)
    p.add_argument("--batch", type=int, default=2048)
    a = p.parse_args()
    a.reports.mkdir(parents=True, exist_ok=True)

    df = pl.read_parquet(a.processed / "interactions.parquet")
    n_users = pl.read_parquet(a.processed / "users.parquet").height
    n_items = pl.read_parquet(a.processed / "items.parquet").height
    feats = load_features(a.processed / "features", NAMES)
    feats["content_all"] = np.concatenate([feats[n] for n in NAMES], axis=1) / np.sqrt(len(NAMES))

    for split, hist in [("val", ["train"]), ("test", ["train", "val"])]:
        H = history_matrix(df, n_users, n_items, hist)
        t = df.filter(pl.col("split") == split).sort("uid")
        target = np.zeros(n_users, dtype=np.int64)
        target[t["uid"].to_numpy()] = t["iid"].to_numpy()
        pop = np.asarray(H.sum(0)).ravel()
        head = np.zeros(n_items, dtype=bool)
        head[np.argsort(-pop)[: int(0.2 * n_items)]] = True

        models = {"popularity": popularity(H), "itemknn": itemknn(H)}
        for n, E in feats.items():
            models[f"content:{n}"] = content_profile(H, E)

        rows = {}
        for name, fn in models.items():
            rows[name] = evaluate(fn, target, head, a.batch)
            print(split, name, round(rows[name]["recall@10"], 4), flush=True)
        print(f"\n== {split} (history: {'+'.join(hist)}, random recall@10 = {10 / n_items:.5f}) ==")
        show(rows)
        (a.reports / f"baselines_{split}.json").write_text(json.dumps(rows, indent=2))


if __name__ == "__main__":
    main()
EOF
cat > $ROOT/scripts/day4_twotower.py <<'EOF'
import argparse
import copy
import json
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F

from vidreclens.config import PROCESSED, REPORTS
from vidreclens.data.history import padded_history
from vidreclens.eval.metrics import evaluate
from vidreclens.features.io import load_features
from vidreclens.models.baselines import history_matrix, mask_history
from vidreclens.models.twotower import TwoTower

ALL = ["clip_vitb32_covers", "videomae", "cliprn50", "bgem3"]
NO_VM = [n for n in ALL if n != "videomae"]
CONFIGS = {
    "id_only": ([], True),
    "content_only": (NO_VM, False),
    "hybrid_3mod": (NO_VM, True),
    "hybrid_all4": (ALL, True),
}
COLS = ["recall@10", "ndcg@10", "recall@50", "ndcg@50", "recall@10_head", "recall@10_tail"]


def table(rows):
    print(f"{'model':<26}" + "".join(f"{c:>16}" for c in COLS))
    for name, r in rows.items():
        print(f"{name:<26}" + "".join(f"{r[c]:>16.4f}" for c in COLS))


def split_ctx(df, split, hist, n_users, n_items, max_len, dev):
    H = history_matrix(df, n_users, n_items, hist)
    t = df.filter(pl.col("split") == split).sort("uid")
    target = np.zeros(n_users, dtype=np.int64)
    target[t["uid"].to_numpy()] = t["iid"].to_numpy()
    pop = np.asarray(H.sum(0)).ravel()
    head = np.zeros(n_items, dtype=bool)
    head[np.argsort(-pop)[: int(0.2 * n_items)]] = True
    P = torch.as_tensor(padded_history(df, hist, n_users, n_items, max_len), device=dev)
    return {"H": H, "target": target, "head": head, "P": P}


def score_fn(model, ctx, pad):
    model.eval()
    with torch.no_grad():
        IR = model.encode_items()

    def fn(idx):
        hist = ctx["P"][torch.as_tensor(idx, device=IR.device)]
        with torch.no_grad():
            s = (model.encode_users(IR, hist, hist != pad) @ IR[:-1].T).float().cpu().numpy()
        return mask_history(s, ctx["H"], idx)

    return fn


def fit(name, cfg, feats, n_items, tr, a, dev, epochs=None):
    names, use_id = cfg
    select = epochs is None
    n_ep = epochs or a.epochs
    torch.manual_seed(a.seed)
    model = TwoTower({k: feats[k] for k in names}, n_items, a.dim, use_id, a.id_drop, a.mod_drop).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
    pad = n_items
    P = tr["P"]
    ru, rp = (torch.as_tensor(x, device=dev) for x in np.nonzero(P.cpu().numpy() != pad))
    ar = torch.arange(P.size(1), device=dev)
    kval = min(a.val_users, len(tr["target"]))
    best, best_state, best_ep, bad = -1.0, None, 0, 0
    t0 = time.time()
    for ep in range(1, n_ep + 1):
        model.train()
        perm = torch.randperm(len(ru), device=dev)
        total = torch.zeros((), device=dev)
        for b in range(0, len(perm), a.batch):
            sel = perm[b : b + a.batch]
            u, p = ru[sel], rp[sel]
            hist = P[u]
            target = hist.gather(1, p[:, None]).squeeze(1)
            keep = (hist != pad) & (ar[None, :] != p[:, None])
            IR = model.encode_items()
            uv = model.encode_users(IR, hist, keep)
            logits = model.log_scale.exp().clamp(max=100.0) * (uv @ IR.T)
            logits = logits.scatter(1, torch.where(keep, hist, torch.full_like(hist, pad)), float("-inf"))
            loss = F.cross_entropy(logits, target)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            total += loss.detach() * len(sel)
        if not select:
            if ep % 5 == 0 or ep == n_ep:
                print(f"{name} refit ep{ep} loss {total.item() / len(ru):.4f} ({time.time() - t0:.0f}s)", flush=True)
            continue
        if ep % a.eval_every == 0 or ep == n_ep:
            r = evaluate(score_fn(model, tr, pad), tr["target"][:kval], tr["head"], a.eval_batch)["recall@10"]
            print(f"{name} ep{ep} loss {total.item() / len(ru):.4f} val_r@10 {r:.4f} ({time.time() - t0:.0f}s)", flush=True)
            if r > best:
                best, best_state, best_ep, bad = r, copy.deepcopy(model.state_dict()), ep, 0
            else:
                bad += 1
                if bad >= a.patience:
                    break
    if select:
        model.load_state_dict(best_state)
    return model


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--processed", type=Path, default=PROCESSED)
    p.add_argument("--reports", type=Path, default=REPORTS)
    p.add_argument("--configs", nargs="+", default=list(CONFIGS), choices=list(CONFIGS))
    p.add_argument("--epochs", type=int, default=40)
    p.add_argument("--batch", type=int, default=2048)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--wd", type=float, default=1e-5)
    p.add_argument("--dim", type=int, default=128)
    p.add_argument("--id-drop", dest="id_drop", type=float, default=0.3)
    p.add_argument("--mod-drop", dest="mod_drop", type=float, default=0.2)
    p.add_argument("--max-len", dest="max_len", type=int, default=50)
    p.add_argument("--eval-every", dest="eval_every", type=int, default=2)
    p.add_argument("--patience", type=int, default=4)
    p.add_argument("--val-users", dest="val_users", type=int, default=10000)
    p.add_argument("--eval-batch", dest="eval_batch", type=int, default=2048)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--refit", nargs="*", default=None)
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = p.parse_args()
    a.reports.mkdir(parents=True, exist_ok=True)
    dev = torch.device(a.device)

    df = pl.read_parquet(a.processed / "interactions.parquet")
    n_users = pl.read_parquet(a.processed / "users.parquet").height
    n_items = pl.read_parquet(a.processed / "items.parquet").height
    feats = {k: torch.as_tensor(v, device=dev) for k, v in load_features(a.processed / "features", ALL).items()}
    val = split_ctx(df, "val", ["train"], n_users, n_items, a.max_len, dev)
    test = split_ctx(df, "test", ["train", "val"], n_users, n_items, a.max_len, dev)

    if a.refit is not None:
        res = {}
        for kv in a.refit:
            name, ep = kv.split("=")
            model = fit(name, CONFIGS[name], feats, n_items, test, a, dev, epochs=int(ep))
            res[f"refit:{name}"] = evaluate(score_fn(model, test, n_items), test["target"], test["head"], a.eval_batch)
            print(name, "refit test r@10", round(res[f"refit:{name}"]["recall@10"], 4), flush=True)
        rows = {}
        for fname in ("baselines_test.json", "twotower_test.json"):
            f = a.reports / fname
            if f.exists():
                prev = json.loads(f.read_text())
                keep = ("popularity", "itemknn", "content:content_all") if fname.startswith("baselines") else tuple(prev)
                rows |= {k: prev[k] for k in keep if k in prev}
        rows |= res
        print("\n== test (refit on train+val) ==")
        table(rows)
        (a.reports / "twotower_test_refit.json").write_text(json.dumps(res, indent=2))
        return

    out = {"val": {}, "test": {}}
    for name in a.configs:
        model = fit(name, CONFIGS[name], feats, n_items, val, a, dev)
        for split, ctx in (("val", val), ("test", test)):
            out[split][f"twotower:{name}"] = evaluate(score_fn(model, ctx, n_items), ctx["target"], ctx["head"], a.eval_batch)
        torch.save(model.state_dict(), a.reports / f"twotower_{name}.pt")
        print(name, "test r@10", round(out["test"][f"twotower:{name}"]["recall@10"], 4), flush=True)

    for split in ("val", "test"):
        rows = dict(out[split])
        f = a.reports / f"baselines_{split}.json"
        if f.exists():
            base = json.loads(f.read_text())
            rows = {k: base[k] for k in ("popularity", "itemknn", "content:content_all") if k in base} | rows
        print(f"\n== {split} ==")
        table(rows)
        (a.reports / f"twotower_{split}.json").write_text(json.dumps(out[split], indent=2))


if __name__ == "__main__":
    main()
EOF
cat > $ROOT/scripts/day5_bias.py <<'EOF'
import argparse
import json
from pathlib import Path

import numpy as np
import polars as pl
import torch

from day4_twotower import ALL, CONFIGS, fit, score_fn, split_ctx
from vidreclens.config import PROCESSED, REPORTS
from vidreclens.data.eda import gini
from vidreclens.eval.metrics import summarize, tie_aware_rank
from vidreclens.features.io import load_features
from vidreclens.models.baselines import itemknn

COLS = ["recall@10", "ndcg@10", "recall@50", "recall@10_head", "recall@10_tail", "coverage@10", "head_share@10", "gini@10"]


def run(fn, ctx, n_users, n_items, batch, k=10):
    target = ctx["target"][:n_users]
    ranks = np.empty(n_users)
    counts = np.zeros(n_items)
    for a0 in range(0, n_users, batch):
        idx = np.arange(a0, min(a0 + batch, n_users))
        s = fn(idx)
        ranks[idx] = tie_aware_rank(s, target[idx])
        top = np.argpartition(-s, k, axis=1)[:, :k]
        counts += np.bincount(top.ravel(), minlength=n_items)
    out = summarize(ranks, ctx["head"][target])
    out["coverage@10"] = float((counts > 0).mean())
    out["head_share@10"] = float(counts[ctx["head"]].sum() / counts.sum())
    out["gini@10"] = gini(counts)
    return out


def sweep(model, ctx, alphas, n_users, n_items, batch):
    base = score_fn(model, ctx, n_items)
    scale = min(model.log_scale.exp().item(), 100.0)
    logpop = np.log1p(np.asarray(ctx["H"].sum(0)).ravel()).astype(np.float32)
    rows = {}
    for al in alphas:
        rows[al] = run(lambda idx, al=al: base(idx) * scale - al * logpop, ctx, n_users, n_items, batch)
        print(f"alpha {al}: recall@10 {rows[al]['recall@10']:.4f} tail {rows[al]['recall@10_tail']:.4f} coverage {rows[al]['coverage@10']:.3f}", flush=True)
    return rows


def table(rows):
    print(f"{'model':<14}" + "".join(f"{c:>16}" for c in COLS))
    for name, r in rows.items():
        print(f"{name:<14}" + "".join(f"{r[c]:>16.4f}" for c in COLS))


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--processed", type=Path, default=PROCESSED)
    p.add_argument("--reports", type=Path, default=REPORTS)
    p.add_argument("--config", default="hybrid_all4", choices=list(CONFIGS))
    p.add_argument("--epochs", type=int, default=22)
    p.add_argument("--alphas", type=float, nargs="+", default=[0.0, 0.25, 0.5, 0.75, 1.0])
    p.add_argument("--tol", type=float, default=0.03)
    p.add_argument("--batch", type=int, default=2048)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--wd", type=float, default=1e-5)
    p.add_argument("--dim", type=int, default=128)
    p.add_argument("--id-drop", dest="id_drop", type=float, default=0.3)
    p.add_argument("--mod-drop", dest="mod_drop", type=float, default=0.2)
    p.add_argument("--max-len", dest="max_len", type=int, default=50)
    p.add_argument("--val-users", dest="val_users", type=int, default=30000)
    p.add_argument("--eval-batch", dest="eval_batch", type=int, default=2048)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = p.parse_args()
    assert a.alphas[0] == 0.0
    a.reports.mkdir(parents=True, exist_ok=True)
    dev = torch.device(a.device)

    df = pl.read_parquet(a.processed / "interactions.parquet")
    n_users = pl.read_parquet(a.processed / "users.parquet").height
    n_items = pl.read_parquet(a.processed / "items.parquet").height
    feats = {k: torch.as_tensor(v, device=dev) for k, v in load_features(a.processed / "features", ALL).items()}
    val = split_ctx(df, "val", ["train"], n_users, n_items, a.max_len, dev)
    test = split_ctx(df, "test", ["train", "val"], n_users, n_items, a.max_len, dev)

    m1 = fit(a.config, CONFIGS[a.config], feats, n_items, val, a, dev, epochs=a.epochs)
    torch.save(m1.state_dict(), a.reports / f"twotower_{a.config}_trainonly.pt")
    kval = min(a.val_users, n_users)
    val_rows = sweep(m1, val, a.alphas, kval, n_items, a.eval_batch)
    ok = [al for al in a.alphas if val_rows[al]["recall@10"] >= (1 - a.tol) * val_rows[0.0]["recall@10"]]
    chosen = max(ok)

    m2 = fit(a.config, CONFIGS[a.config], feats, n_items, test, a, dev, epochs=a.epochs)
    torch.save(m2.state_dict(), a.reports / f"twotower_{a.config}_refit.pt")
    test_rows = sweep(m2, test, a.alphas, n_users, n_items, a.eval_batch)
    knn = run(itemknn(test["H"]), test, n_users, n_items, a.eval_batch)

    print(f"\n== val (train-only model, first {kval} users) ==")
    table({f"a={al}": r for al, r in val_rows.items()})
    print("\n== test (refit on train+val, all users) ==")
    table({"itemknn": knn} | {f"a={al}": r for al, r in test_rows.items()})
    print(f"\nchosen alpha = {chosen} (largest alpha with val recall@10 within {a.tol:.0%} of alpha=0)")
    r0, rc = test_rows[0.0], test_rows[chosen]
    print(f"test at chosen alpha vs alpha=0: recall@10 {r0['recall@10']:.4f} -> {rc['recall@10']:.4f}, tail recall@10 {r0['recall@10_tail']:.4f} -> {rc['recall@10_tail']:.4f}, coverage@10 {r0['coverage@10']:.3f} -> {rc['coverage@10']:.3f}")
    (a.reports / "bias_val.json").write_text(json.dumps({str(k): v for k, v in val_rows.items()}, indent=2))
    (a.reports / "bias_test.json").write_text(json.dumps({"itemknn": knn} | {str(k): v for k, v in test_rows.items()}, indent=2))


if __name__ == "__main__":
    main()
EOF
cat > $ROOT/scripts/day6_ab.py <<'EOF'
import argparse
import json
import math
from pathlib import Path

import numpy as np
import polars as pl
import torch

from day4_twotower import ALL, fit, score_fn, split_ctx, CONFIGS
from vidreclens.config import PROCESSED, REPORTS
from vidreclens.eval.metrics import tie_aware_rank
from vidreclens.features.io import load_features
from vidreclens.models.baselines import itemknn
from vidreclens.models.twotower import TwoTower

ZA, ZB = 1.959964, 0.841621


def user_ranks(fn, target, n_users, batch):
    ranks = np.empty(n_users)
    for a0 in range(0, n_users, batch):
        idx = np.arange(a0, min(a0 + batch, n_users))
        ranks[idx] = tie_aware_rank(fn(idx), target[idx])
    return ranks


def outcomes(r, k=10):
    hit = r <= k
    return hit.astype(np.float64), np.where(hit, 1 / np.log2(r + 1), 0.0)


def pvalue(z):
    return math.erfc(abs(z) / math.sqrt(2))


def mean_ci(x):
    m = float(x.mean())
    h = 1.96 * float(x.std(ddof=1)) / math.sqrt(len(x))
    return m, m - h, m + h


def paired(a, b):
    d = b - a
    m = float(d.mean())
    se = float(d.std(ddof=1)) / math.sqrt(len(d))
    z = m / se if se > 0 else 0.0
    return m, m - 1.96 * se, m + 1.96 * se, pvalue(z)


def required_n(p1, p2):
    return (ZA + ZB) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2


def simulate(pairs, n_arm, sims, rng):
    n = len(next(iter(pairs.values()))[0])
    sig = {k: np.zeros(sims, dtype=bool) for k in pairs}
    lift = {k: np.zeros(sims) for k in pairs}
    for s in range(sims):
        idx = rng.permutation(n)[: 2 * n_arm]
        c, t = idx[:n_arm], idx[n_arm:]
        for k, (ha, hb) in pairs.items():
            pc, pt = ha[c].mean(), hb[t].mean()
            se = math.sqrt((pc * (1 - pc) + pt * (1 - pt)) / n_arm)
            sig[k][s] = se > 0 and abs(pt - pc) / se > ZA
            lift[k][s] = (pt - pc) / pc if pc > 0 else 0.0
    return {
        k: {
            "power": float(sig[k].mean()),
            "lift_p5": float(np.percentile(lift[k], 5)),
            "lift_median": float(np.median(lift[k])),
            "lift_p95": float(np.percentile(lift[k], 95)),
        }
        for k in pairs
    }


def segments(labels, names, hits, a_key, b_key):
    rows = []
    for i, nm in enumerate(names):
        m = labels == i
        if not m.any():
            continue
        d = paired(hits[a_key][m], hits[b_key][m]) if m.sum() > 1 else (0.0, 0.0, 0.0, 1.0)
        rows.append({"segment": nm, "users": int(m.sum()), **{k: float(v[m].mean()) for k, v in hits.items()}, "diff": d[0], "p": d[3]})
    return rows


def print_segments(title, rows, keys, a_key, b_key):
    print(f"\n{title}")
    print(f"{'segment':<10}{'users':>9}" + "".join(f"{k:>12}" for k in keys) + f"{b_key + '-' + a_key:>20}{'p':>10}")
    for r in rows:
        print(f"{r['segment']:<10}{r['users']:>9}" + "".join(f"{r[k]:>12.4f}" for k in keys) + f"{r['diff']:>+20.4f}{r['p']:>10.4f}")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--processed", type=Path, default=PROCESSED)
    p.add_argument("--reports", type=Path, default=REPORTS)
    p.add_argument("--ckpt", type=Path, default=None)
    p.add_argument("--config", default="hybrid_all4", choices=list(CONFIGS))
    p.add_argument("--alpha", type=float, default=0.75)
    p.add_argument("--epochs", type=int, default=22)
    p.add_argument("--sims", type=int, default=1000)
    p.add_argument("--arm-sizes", dest="arm_sizes", type=int, nargs="+", default=[5000, 10000, 20000, 35000])
    p.add_argument("--batch", type=int, default=2048)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--wd", type=float, default=1e-5)
    p.add_argument("--dim", type=int, default=128)
    p.add_argument("--id-drop", dest="id_drop", type=float, default=0.3)
    p.add_argument("--mod-drop", dest="mod_drop", type=float, default=0.2)
    p.add_argument("--max-len", dest="max_len", type=int, default=50)
    p.add_argument("--val-users", dest="val_users", type=int, default=30000)
    p.add_argument("--eval-batch", dest="eval_batch", type=int, default=2048)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = p.parse_args()
    a.reports.mkdir(parents=True, exist_ok=True)
    dev = torch.device(a.device)
    ckpt = a.ckpt or a.reports / f"twotower_{a.config}_refit.pt"

    df = pl.read_parquet(a.processed / "interactions.parquet")
    n_users = pl.read_parquet(a.processed / "users.parquet").height
    n_items = pl.read_parquet(a.processed / "items.parquet").height
    feats = {k: torch.as_tensor(v, device=dev) for k, v in load_features(a.processed / "features", ALL).items()}
    test = split_ctx(df, "test", ["train", "val"], n_users, n_items, a.max_len, dev)

    names, use_id = CONFIGS[a.config]
    if ckpt.exists():
        model = TwoTower({k: feats[k] for k in names}, n_items, a.dim, use_id, a.id_drop, a.mod_drop).to(dev)
        model.load_state_dict(torch.load(ckpt, map_location=dev))
        print(f"loaded {ckpt}")
    else:
        print(f"{ckpt} not found, training {a.config} on train+val for {a.epochs} epochs")
        model = fit(a.config, CONFIGS[a.config], feats, n_items, test, a, dev, epochs=a.epochs)

    base = score_fn(model, test, n_items)
    scale = min(model.log_scale.exp().item(), 100.0)
    logpop = np.log1p(np.asarray(test["H"].sum(0)).ravel()).astype(np.float32)
    tag = f"tt_a{a.alpha:g}"
    policies = {
        "itemknn": itemknn(test["H"]),
        "tt_a0": lambda idx: base(idx) * scale,
        tag: lambda idx: base(idx) * scale - a.alpha * logpop,
    }
    ranks = {}
    for k, fn in policies.items():
        ranks[k] = user_ranks(fn, test["target"], n_users, a.eval_batch)
        print(f"scored {k}", flush=True)
    hit, ndcg = {}, {}
    for k, r in ranks.items():
        hit[k], ndcg[k] = outcomes(r)

    res = {"n_users": n_users, "alpha": a.alpha}
    print(f"\n== 1. Policy metrics on {n_users} test users (95% CI) ==")
    res["metrics"] = {}
    for k in policies:
        h, n_ = mean_ci(hit[k]), mean_ci(ndcg[k])
        res["metrics"][k] = {"hit@10": h, "ndcg@10": n_}
        print(f"{k:<10} hit@10 {h[0]:.4f} [{h[1]:.4f}, {h[2]:.4f}]   ndcg@10 {n_[0]:.4f} [{n_[1]:.4f}, {n_[2]:.4f}]")

    comps = [("tt_a0", tag), ("itemknn", tag), ("itemknn", "tt_a0")]
    print("\n== 2. Paired comparison, same users under both policies (B minus A) ==")
    res["paired"] = {}
    for ka, kb in comps:
        dh, dn = paired(hit[ka], hit[kb]), paired(ndcg[ka], ndcg[kb])
        only_b = int(((hit[kb] == 1) & (hit[ka] == 0)).sum())
        only_a = int(((hit[kb] == 0) & (hit[ka] == 1)).sum())
        res["paired"][f"{ka}->{kb}"] = {"hit": dh, "ndcg": dn, "only_b": only_b, "only_a": only_a}
        print(f"A={ka:<8} B={kb:<8} hit@10 {dh[0]:+.4f} [{dh[1]:+.4f}, {dh[2]:+.4f}] p={dh[3]:.4f}   ndcg@10 {dn[0]:+.4f} [{dn[1]:+.4f}, {dn[2]:+.4f}] p={dn[3]:.4f}   hits only A/only B: {only_a}/{only_b}")

    sizes = sorted({s for s in a.arm_sizes if s <= n_users // 2} | {n_users // 2})
    pairs = {f"{ka}->{kb}": (hit[ka], hit[kb]) for ka, kb in comps}
    rng = np.random.default_rng(a.seed)
    print(f"\n== 3. Simulated A/B test: users split at random, each sees only its arm's policy ({a.sims} simulations per size) ==")
    print(f"{'users/arm':>10}  {'comparison':<22}{'power':>8}{'lift p5':>10}{'lift med':>10}{'lift p95':>10}")
    res["ab"] = {}
    for s in sizes:
        out = simulate(pairs, s, a.sims, rng)
        res["ab"][s] = out
        for k, v in out.items():
            print(f"{s:>10}  {k:<22}{v['power']:>8.2f}{v['lift_p5']:>+10.1%}{v['lift_median']:>+10.1%}{v['lift_p95']:>+10.1%}")

    p1, p2 = float(hit["tt_a0"].mean()), float(hit[tag].mean())
    print(f"\n== 4. Users needed per arm (80% power, 5% two-sided) when the baseline hit rate is {p1:.4f} ==")
    res["required"] = {}
    for rel in (0.02, 0.05, 0.10, 0.20):
        n_req = required_n(p1, p1 * (1 + rel))
        res["required"][rel] = n_req
        print(f"relative lift {rel:>4.0%}: {n_req:>10,.0f}")
    obs = required_n(p1, p2)
    res["required_observed"] = obs
    print(f"observed lift {p2 / p1 - 1:+.1%} ({p1:.4f} -> {p2:.4f}): {obs:,.0f} per arm; available {n_users // 2:,} per arm")

    hist_n = np.asarray(test["H"].sum(1)).ravel()
    hl = np.select([hist_n <= 4, hist_n <= 6, hist_n <= 10], [0, 1, 2], 3)
    tpop = np.asarray(test["H"].sum(0)).ravel()[test["target"]]
    tl = np.select([tpop == 0, tpop <= 4, tpop <= 19, tpop <= 99], [0, 1, 2, 3], 4)
    keys = ["itemknn", "tt_a0", tag]
    res["segments"] = {
        "history_length": segments(hl, ["4", "5-6", "7-10", "11+"], {k: hit[k] for k in keys}, "itemknn", tag),
        "target_popularity": segments(tl, ["0", "1-4", "5-19", "20-99", "100+"], {k: hit[k] for k in keys}, "itemknn", tag),
    }
    print_segments("== 5a. hit@10 by length of the user's history ==", res["segments"]["history_length"], keys, "itemknn", tag)
    print_segments("== 5b. hit@10 by how many past interactions the target video had (0 = cold item) ==", res["segments"]["target_popularity"], keys, "itemknn", tag)

    (a.reports / "day6_ab.json").write_text(json.dumps(res, indent=2, default=float))
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(6, 4))
    for k in pairs:
        ax.plot(sizes, [res["ab"][s][k]["power"] for s in sizes], marker="o", label=k)
    ax.axhline(0.8, color="grey", linestyle="--", linewidth=1)
    ax.set_xlabel("users per arm")
    ax.set_ylabel("share of simulations with p < 0.05")
    ax.set_title("Simulated A/B test power")
    ax.legend()
    fig.tight_layout()
    fig.savefig(a.reports / "ab_power.png", dpi=150)
    print("\nsaved day6_ab.json and ab_power.png")


if __name__ == "__main__":
    main()
EOF
cat > $ROOT/scripts/day7_clusters.py <<'EOF'
import argparse
import json
import re
from pathlib import Path

import numpy as np
import polars as pl
import torch

from day4_twotower import ALL, CONFIGS, fit, score_fn, split_ctx
from vidreclens.config import PROCESSED, REPORTS
from vidreclens.eval.metrics import tie_aware_rank
from vidreclens.features.io import load_features
from vidreclens.models.baselines import itemknn
from vidreclens.models.twotower import TwoTower

CONTENT = [n for n in ALL if n != "videomae"]
TITLES = Path("/kaggle/working/data/MicroLens-100k_title_en.csv")


def load_titles(path, items):
    titles = {}
    if path.exists():
        for line in path.open(encoding="utf-8", errors="ignore"):
            m = re.match(r"\s*(\d+)\s*,\s*(.*)", line)
            if m:
                titles[int(m.group(1))] = m.group(2).strip()
    return [titles.get(int(i), "") for i in items]


def cluster_items(feats, k, seed):
    from sklearn.cluster import KMeans
    from sklearn.decomposition import PCA

    X = np.concatenate([feats[n] for n in CONTENT], axis=1)
    Z = PCA(n_components=64, random_state=seed).fit_transform(X)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True).clip(1e-8)
    return KMeans(n_clusters=k, n_init=5, random_state=seed).fit(Z).labels_, Z


def score_policy(fn, target, n_users, n_items, batch, k=10):
    ranks = np.empty(n_users)
    counts = np.zeros(n_items)
    for a0 in range(0, n_users, batch):
        idx = np.arange(a0, min(a0 + batch, n_users))
        s = fn(idx)
        ranks[idx] = tie_aware_rank(s, target[idx])
        counts += np.bincount(np.argpartition(-s, k, axis=1)[:, :k].ravel(), minlength=n_items)
    return ranks, counts


def short(t, n=60):
    return t if len(t) <= n else t[: n - 1] + "…"


def distinctive_words(titles, labels, K, top=8, min_count=15):
    from collections import Counter

    overall, per = Counter(), [Counter() for _ in range(K)]
    for t, c in zip(titles, labels):
        w = set(re.findall(r"[a-z]{3,}", t.lower()))
        overall.update(w)
        per[c].update(w)
    n, sizes = len(titles), np.bincount(labels, minlength=K)
    out = []
    for c in range(K):
        sc = {w: (k / sizes[c]) / (overall[w] / n) for w, k in per[c].items() if overall[w] >= min_count and k >= 5}
        out.append([w for w, _ in sorted(sc.items(), key=lambda kv: -kv[1])[:top]])
    return out


def name_prompt(rows, words):
    lines = [
        "Below, for each cluster of short videos, are example titles (machine-translated from Chinese, so the wording is rough) and words that are unusually frequent in the cluster's titles.",
        "For each cluster write exactly one line: 'C<number>: <2 to 4 word topic>'. The topic must be justified by the titles or words. If they share no clear topic, write 'C<number>: mixed'. Output only these lines.",
        "",
    ]
    for r, w in zip(rows, words):
        lines.append(f"C{r['cluster']} | words: {', '.join(w) or 'none'} | titles: " + "; ".join(short(t, 70) for t in r["titles"][:5]))
    return "\n".join(lines)


def rewrite_prompt(facts):
    return (
        "Rewrite the facts below as one short paragraph of at most 5 sentences for a product manager. "
        "Rules: use only the facts below; every number you write must appear in the facts; refer to clusters as C0, C1 and so on, never as categories; write each change as 'from X to Y'; do not use evaluative words such as significantly or notably; do not explain causes; do not mention genres, users' feelings or anything not stated.\n\nFacts:\n"
        + "\n".join("- " + f for f in facts)
    )


def unsupported_numbers(text, facts):
    ok = set(re.findall(r"\d+(?:\.\d+)?", facts)) | {"10"}
    text = re.sub(r"(?<=\d),(?=\d{3}(?!\d))", "", text)
    return sorted(set(re.findall(r"\d+(?:\.\d+)?", text)) - ok)


def clean_names(text):
    lines = [re.sub(r"^(C\d+):\s*C?#\s*", r"\1: ", ln.strip()) for ln in text.splitlines() if ln.strip()]
    return "\n".join(lines)


def run_llm(name, prompts, dev):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16 if dev.type == "cuda" else torch.float32).to(dev).eval()
    outs = []
    for pr in prompts:
        msgs = [
            {"role": "system", "content": "You are a careful analyst. You only use facts given to you and never invent numbers or causes."},
            {"role": "user", "content": pr},
        ]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = tok(text, return_tensors="pt").to(dev)
        with torch.inference_mode():
            out = model.generate(**inp, max_new_tokens=400, do_sample=False)
        outs.append(tok.decode(out[0][inp["input_ids"].shape[1] :], skip_special_tokens=True).strip())
    return outs


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--processed", type=Path, default=PROCESSED)
    p.add_argument("--reports", type=Path, default=REPORTS)
    p.add_argument("--titles", type=Path, default=TITLES)
    p.add_argument("--ckpt", type=Path, default=None)
    p.add_argument("--config", default="hybrid_all4", choices=list(CONFIGS))
    p.add_argument("--alpha", type=float, default=0.75)
    p.add_argument("--k", type=int, default=12)
    p.add_argument("--llm", default="Qwen/Qwen2.5-1.5B-Instruct")
    p.add_argument("--epochs", type=int, default=22)
    p.add_argument("--batch", type=int, default=2048)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--wd", type=float, default=1e-5)
    p.add_argument("--dim", type=int, default=128)
    p.add_argument("--id-drop", dest="id_drop", type=float, default=0.3)
    p.add_argument("--mod-drop", dest="mod_drop", type=float, default=0.2)
    p.add_argument("--max-len", dest="max_len", type=int, default=50)
    p.add_argument("--val-users", dest="val_users", type=int, default=30000)
    p.add_argument("--eval-batch", dest="eval_batch", type=int, default=2048)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = p.parse_args()
    a.reports.mkdir(parents=True, exist_ok=True)
    dev = torch.device(a.device)
    ckpt = a.ckpt or a.reports / f"twotower_{a.config}_refit.pt"

    df = pl.read_parquet(a.processed / "interactions.parquet")
    n_users = pl.read_parquet(a.processed / "users.parquet").height
    items_df = pl.read_parquet(a.processed / "items.parquet").sort("iid")
    n_items = items_df.height
    stats = pl.read_parquet(a.processed / "item_stats.parquet").sort("iid")
    feats_np = load_features(a.processed / "features", ALL)
    feats = {k: torch.as_tensor(v, device=dev) for k, v in feats_np.items()}
    test = split_ctx(df, "test", ["train", "val"], n_users, n_items, a.max_len, dev)
    titles = load_titles(a.titles, items_df["item"].to_numpy())

    labels, Z = cluster_items(feats_np, a.k, a.seed)
    K = a.k
    words = distinctive_words(titles, labels, K)

    names, use_id = CONFIGS[a.config]
    if ckpt.exists():
        model = TwoTower({k: feats[k] for k in names}, n_items, a.dim, use_id, a.id_drop, a.mod_drop).to(dev)
        model.load_state_dict(torch.load(ckpt, map_location=dev))
        print(f"loaded {ckpt}")
    else:
        print(f"{ckpt} not found, training {a.config} on train+val for {a.epochs} epochs")
        model = fit(a.config, CONFIGS[a.config], feats, n_items, test, a, dev, epochs=a.epochs)

    base = score_fn(model, test, n_items)
    scale = min(model.log_scale.exp().item(), 100.0)
    logpop = np.log1p(np.asarray(test["H"].sum(0)).ravel()).astype(np.float32)
    policies = {
        "itemknn": itemknn(test["H"]),
        "tt_a0": lambda idx: base(idx) * scale,
        "adj": lambda idx: base(idx) * scale - a.alpha * logpop,
    }
    target = test["target"]
    hit, counts = {}, {}
    for k, fn in policies.items():
        r, c = score_policy(fn, target, n_users, n_items, a.eval_batch)
        hit[k], counts[k] = (r <= 10).astype(float), c
        print(f"scored {k}", flush=True)

    tl = labels[target]
    demand = np.bincount(tl, minlength=K) / n_users
    expo = {k: np.bincount(labels, weights=c, minlength=K) / c.sum() for k, c in counts.items()}
    tvd = {k: float(0.5 * np.abs(expo[k] - demand).sum()) for k in policies}
    catalog = np.bincount(labels, minlength=K) / n_items
    tvdc = {k: float(0.5 * np.abs(expo[k] - catalog).sum()) for k in policies}
    hitc = {k: np.array([hit[k][tl == c].mean() if (tl == c).any() else np.nan for c in range(K)]) for k in policies}
    users_c = np.bincount(tl, minlength=K)
    log_views = stats["log_views"].to_numpy()
    like_rate = stats["like_rate"].to_numpy()
    tail = ~test["head"]

    rows = []
    for c in range(K):
        idx = np.where(labels == c)[0]
        cen = Z[idx].mean(0)
        rep = idx[np.argsort(-(Z[idx] @ cen))[:5]]
        rows.append(
            {
                "cluster": c,
                "items": int(len(idx)),
                "catalog_pct": 100 * len(idx) / n_items,
                "demand_pct": 100 * float(demand[c]),
                "tail_item_pct": 100 * float(tail[idx].mean()),
                "mean_log_views": float(log_views[idx].mean()),
                "mean_like_rate": float(like_rate[idx].mean()),
                "users": int(users_c[c]),
                "expo_pct": {k: 100 * float(expo[k][c]) for k in policies},
                "ratio": {k: float(expo[k][c] / max(demand[c], 1e-9)) for k in policies},
                "hit": {k: float(hitc[k][c]) for k in policies},
                "titles": [titles[i] or f"(item {items_df['item'][int(i)]})" for i in rep],
                "words": words[c],
            }
        )

    print(f"\n== Clusters: composition and demand (k={K}, KMeans on PCA of cover + title + CLIP RN50 embeddings) ==")
    print(f"{'cluster':<8}{'items':>7}{'catalog%':>10}{'demand%':>9}{'tail items%':>13}{'log views':>11}{'like rate':>11}   example titles")
    for r in rows:
        print(f"C{r['cluster']:<7}{r['items']:>7}{r['catalog_pct']:>10.1f}{r['demand_pct']:>9.1f}{r['tail_item_pct']:>13.1f}{r['mean_log_views']:>11.2f}{r['mean_like_rate']:>11.3f}   " + " | ".join(short(t, 28) for t in r["titles"][:2]))
    print("\n== Distinctive title words by cluster (much more frequent in this cluster than overall) ==")
    for c, w in enumerate(words):
        print(f"C{c:<4}" + ", ".join(w))
    print(f"\n== Policy KPIs by cluster (exposure = share of all top-10 slots; ratio = exposure / demand; hit@10 for users whose next video is in the cluster) ==")
    print(f"{'cluster':<8}{'users':>7}  {'exposure% knn / tt / adj':<28}{'ratio tt / adj':<18}{'hit@10 knn / tt / adj'}")
    for r in rows:
        e, q, h = r["expo_pct"], r["ratio"], r["hit"]
        print(f"C{r['cluster']:<7}{r['users']:>7}  {e['itemknn']:>6.1f} {e['tt_a0']:>6.1f} {e['adj']:>6.1f}          {q['tt_a0']:>5.2f} {q['adj']:>5.2f}     {h['itemknn']:.3f} {h['tt_a0']:.3f} {h['adj']:.3f}")
    print(f"\nExposure vs demand (total variation distance, lower = exposure follows what users watch): item-kNN {tvd['itemknn']:.3f} | two-tower {tvd['tt_a0']:.3f} | two-tower alpha {a.alpha:g} {tvd['adj']:.3f}")
    print(f"Exposure vs catalogue share (lower = each cluster gets slots in proportion to its size): item-kNN {tvdc['itemknn']:.3f} | two-tower {tvdc['tt_a0']:.3f} | two-tower alpha {a.alpha:g} {tvdc['adj']:.3f}")

    ok = [r for r in rows if r["demand_pct"] >= 1.0 and r["users"] >= 100]
    over = max(ok, key=lambda r: r["ratio"]["tt_a0"])
    under = min(ok, key=lambda r: r["ratio"]["tt_a0"])
    gain = max(ok, key=lambda r: r["hit"]["adj"] - r["hit"]["tt_a0"])
    loss = min(ok, key=lambda r: r["hit"]["adj"] - r["hit"]["tt_a0"])
    gap = max(ok, key=lambda r: r["hit"]["itemknn"] - r["hit"]["adj"])
    found = [
        f"Without adjustment the two-tower puts {over['ratio']['tt_a0']:.2f}x the demand share into C{over['cluster']} and {under['ratio']['tt_a0']:.2f}x into C{under['cluster']}; with alpha {a.alpha:g} these become {over['ratio']['adj']:.2f}x and {under['ratio']['adj']:.2f}x.",
        f"The distance of exposure from demand {'falls' if tvd['adj'] < tvd['tt_a0'] else 'rises'} from {tvd['tt_a0']:.3f} to {tvd['adj']:.3f}, and its distance from catalogue share {'falls' if tvdc['adj'] < tvdc['tt_a0'] else 'rises'} from {tvdc['tt_a0']:.3f} to {tvdc['adj']:.3f} (item-kNN: {tvd['itemknn']:.3f} and {tvdc['itemknn']:.3f}).",
        f"hit@10 changes most favourably in C{gain['cluster']} ({gain['hit']['tt_a0']:.3f} to {gain['hit']['adj']:.3f}, {gain['users']} users) and most unfavourably in C{loss['cluster']} ({loss['hit']['tt_a0']:.3f} to {loss['hit']['adj']:.3f}, {loss['users']} users).",
        f"item-kNN's largest lead over the adjusted two-tower is in C{gap['cluster']} ({gap['hit']['itemknn']:.3f} vs {gap['hit']['adj']:.3f}, {gap['users']} users).",
    ]
    print("\n== Findings computed directly from the table ==")
    for f in found:
        print("-", f)

    llm = {"names": "", "summary": "", "unsupported_numbers": []}
    if a.llm.lower() != "none":
        try:
            print(f"\n== LLM output ({a.llm}); suggestions only, check against the tables above ==")
            llm["names"], llm["summary"] = run_llm(a.llm, [name_prompt(rows, words), rewrite_prompt(found)], dev)
            llm["names"] = clean_names(llm["names"])
            llm["unsupported_numbers"] = unsupported_numbers(llm["summary"], "\n".join(found))
            print("Suggested cluster names:")
            print(llm["names"])
            print("\nParagraph rewritten from the findings above:")
            print(llm["summary"])
            print("number check:", "OK, every number appears in the findings" if not llm["unsupported_numbers"] else f"WARNING, numbers not in the findings: {llm['unsupported_numbers']}")
        except Exception as e:
            print(f"LLM step skipped: {type(e).__name__}: {str(e)[:200]}")

    out = {"k": K, "alpha": a.alpha, "tvd_demand": tvd, "tvd_catalog": tvdc, "clusters": rows, "findings": found, "llm_model": a.llm, "llm": llm}
    (a.reports / "day7_clusters.json").write_text(json.dumps(out, indent=2, default=float))
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    x = np.arange(K)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - 0.27, [r["demand_pct"] for r in rows], 0.27, label="demand", color="#555555")
    ax.bar(x, [r["expo_pct"]["tt_a0"] for r in rows], 0.27, label="two-tower", color="#999999")
    ax.bar(x + 0.27, [r["expo_pct"]["adj"] for r in rows], 0.27, label=f"two-tower, alpha {a.alpha:g}", color="#cccccc", edgecolor="#555555")
    ax.set_xticks(x)
    ax.set_xticklabels([f"C{c}" for c in range(K)])
    ax.set_ylabel("% of top-10 slots / % of demand")
    ax.set_title("Exposure vs demand by content cluster")
    ax.legend()
    fig.tight_layout()
    fig.savefig(a.reports / "cluster_exposure.png", dpi=150)
    print("\nsaved day7_clusters.json and cluster_exposure.png")


if __name__ == "__main__":
    main()
EOF
cat > $ROOT/scripts/export_bundle.py <<'EOF'
import argparse
import shutil
from pathlib import Path

import numpy as np
import polars as pl
import torch

from day4_twotower import ALL, CONFIGS
from day7_clusters import TITLES, load_titles
from vidreclens.config import PROCESSED, REPORTS
from vidreclens.data.history import padded_history
from vidreclens.features.io import load_features
from vidreclens.models.twotower import TwoTower
from vidreclens.serve.engine import Recommender
from vidreclens.serve.export import export_bundle


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--processed", type=Path, default=PROCESSED)
    p.add_argument("--reports", type=Path, default=REPORTS)
    p.add_argument("--titles", type=Path, default=TITLES)
    p.add_argument("--out", type=Path, default=Path("/kaggle/working/serving_bundle"))
    p.add_argument("--ckpt", type=Path, default=None)
    p.add_argument("--config", default="hybrid_all4", choices=list(CONFIGS))
    p.add_argument("--alpha", type=float, default=0.75)
    p.add_argument("--dim", type=int, default=128)
    p.add_argument("--id-drop", dest="id_drop", type=float, default=0.3)
    p.add_argument("--mod-drop", dest="mod_drop", type=float, default=0.2)
    p.add_argument("--max-len", dest="max_len", type=int, default=50)
    p.add_argument("--checks", type=int, default=200)
    a = p.parse_args()
    ckpt = a.ckpt or a.reports / f"twotower_{a.config}_refit.pt"
    if not ckpt.exists():
        raise SystemExit(f"{ckpt} not found; run scripts/day5_bias.py first")

    df = pl.read_parquet(a.processed / "interactions.parquet")
    n_users = pl.read_parquet(a.processed / "users.parquet").height
    items_df = pl.read_parquet(a.processed / "items.parquet").sort("iid")
    n_items = items_df.height
    item_ids = items_df["item"].to_numpy()
    feats = {k: torch.as_tensor(v) for k, v in load_features(a.processed / "features", ALL).items()}
    names, use_id = CONFIGS[a.config]
    model = TwoTower({k: feats[k] for k in names}, n_items, a.dim, use_id, a.id_drop, a.mod_drop)
    model.load_state_dict(torch.load(ckpt, map_location="cpu"))
    model.eval()

    seen = df.filter(pl.col("split").is_in(["train", "val"]))["iid"].to_numpy()
    pop = np.bincount(seen, minlength=n_items)
    titles = load_titles(a.titles, item_ids)
    out = export_bundle(model, item_ids, titles, pop, a.out, a.alpha, a.max_len)
    eng = Recommender(out)

    P = padded_history(df, ["train", "val"], n_users, n_items, a.max_len)
    with torch.no_grad():
        IR = model.encode_items()
    logpop = torch.log1p(torch.as_tensor(pop, dtype=torch.float32))
    same, worst = 0, 0.0
    n = min(a.checks, n_users)
    for u in range(n):
        hist = P[u][P[u] != n_items]
        h = torch.as_tensor(hist)[None]
        with torch.no_grad():
            uv = model.encode_users(IR, h, torch.ones_like(h, dtype=torch.bool))[0]
            s = eng.scale * (IR[:-1] @ uv) - a.alpha * logpop
            s[torch.as_tensor(hist)] = -float("inf")
            top = torch.topk(s, 10)
        ref = [int(item_ids[i]) for i in top.indices]
        res = eng.recommend([int(item_ids[i]) for i in hist], 10)
        got = [r["item_id"] for r in res["items"]]
        same += int(got == ref)
        worst = max(worst, float(np.abs(np.array([r["score"] for r in res["items"]]) - top.values.numpy()).max()))
    print(f"parity check on {n} users: identical top-10 for {same}/{n}, largest score difference {worst:.2e}")
    zp = shutil.make_archive(str(out), "zip", out)
    size = sum(f.stat().st_size for f in out.iterdir()) / 1e6
    print(f"bundle {out} ({size:.1f} MB uncompressed) -> {zp}")


if __name__ == "__main__":
    main()
EOF
cat > $ROOT/tests/test_data.py <<'EOF'
import polars as pl

from vidreclens.data.clean import clean_interactions, kcore
from vidreclens.data.eda import gini, summarize
from vidreclens.data.split import leave_last_out
import numpy as np


def make_raw():
    rows = []
    for u in range(1, 21):
        for j in range(8):
            rows.append((u, 100 + (u + j) % 10, 1000 + j))
    rows += [(1, 100, 5000), (1, 100, 5001)]
    rows += [(99, 777, 1), (99, 778, 2)]
    return pl.DataFrame(rows, schema=["user", "item", "ts"], orient="row").lazy()


def test_clean_dedup_and_kcore():
    df, users, items = clean_interactions(make_raw(), 5, 5)
    assert df.group_by("uid").len()["len"].min() >= 5
    assert df.group_by("iid").len()["len"].min() >= 5
    assert df.select(pl.struct("uid", "iid").is_duplicated().any()).item() is False
    assert 99 not in users["user"].to_list()
    assert sorted(df["uid"].unique().to_list()) == list(range(users.height))
    assert sorted(df["iid"].unique().to_list()) == list(range(items.height))


def test_split_is_temporal_and_complete():
    df, _, _ = clean_interactions(make_raw(), 5, 5)
    sp = leave_last_out(df)
    per_user = sp.group_by("uid").agg(
        (pl.col("split") == "test").sum().alias("t"),
        (pl.col("split") == "val").sum().alias("v"),
    )
    assert per_user["t"].to_list() == [1] * per_user.height
    assert per_user["v"].to_list() == [1] * per_user.height
    mx = sp.filter(pl.col("split") == "train").group_by("uid").agg(pl.col("ts").max().alias("m"))
    tmin = sp.filter(pl.col("split") == "test").select("uid", "ts").join(mx, on="uid")
    assert (tmin["ts"] >= tmin["m"]).all()


def test_gini_bounds():
    assert abs(gini(np.ones(100))) < 1e-9
    assert gini(np.array([0] * 99 + [100])) > 0.98


def test_summarize_keys():
    df, _, _ = clean_interactions(make_raw(), 5, 5)
    s = summarize(leave_last_out(df))
    assert 0 < s["sparsity"] < 1 and s["users"] == 20


def test_padded_history_keeps_latest_and_pads():
    from vidreclens.data.history import padded_history

    df = pl.DataFrame(
        {
            "uid": [0, 0, 0, 0, 0, 1, 1, 1],
            "iid": [5, 6, 7, 8, 9, 1, 2, 3],
            "ts": [1, 2, 3, 4, 5, 1, 2, 3],
            "split": ["train"] * 4 + ["val"] + ["train"] * 3,
        }
    )
    P = padded_history(df, ["train"], 3, 10, 3)
    assert P.tolist() == [[6, 7, 8], [1, 2, 3], [10, 10, 10]]
    P2 = padded_history(df, ["train", "val"], 3, 10, 3)
    assert P2[0].tolist() == [7, 8, 9]
EOF
cat > $ROOT/tests/test_serve.py <<'EOF'
import numpy as np
import pytest

torch = pytest.importorskip("torch")
pytest.importorskip("fastapi")
pytest.importorskip("httpx")

from fastapi.testclient import TestClient

from vidreclens.models.twotower import TwoTower
from vidreclens.serve.app import create_app
from vidreclens.serve.engine import Recommender
from vidreclens.serve.export import export_bundle

N, D = 80, 16


@pytest.fixture(scope="module")
def setup(tmp_path_factory):
    torch.manual_seed(0)
    rng = np.random.default_rng(0)
    feats = {"a": torch.randn(N, 24), "b": torch.randn(N, 10)}
    model = TwoTower(feats, N, d=D, use_id=True)
    with torch.no_grad():
        model.id.weight.normal_(0, 0.5)
    item_ids = rng.permutation(np.arange(1000, 1000 + N))
    titles = [f"video {i}, with comma" for i in item_ids]
    pop = rng.integers(0, 50, N)
    out = export_bundle(model, item_ids, titles, pop, tmp_path_factory.mktemp("bundle"), alpha=0.75, max_len=20)
    return model, item_ids, pop, out


def torch_top(model, item_ids, pop, hist_iids, k, alpha, scale):
    model.eval()
    with torch.no_grad():
        IR = model.encode_items()
        h = torch.as_tensor(hist_iids)[None]
        u = model.encode_users(IR, h, torch.ones_like(h, dtype=torch.bool))[0]
        s = scale * (IR[:-1] @ u) - alpha * torch.log1p(torch.as_tensor(pop, dtype=torch.float32))
        s[torch.as_tensor(hist_iids)] = -float("inf")
        top = torch.topk(s, k)
    return [int(item_ids[i]) for i in top.indices], top.values.numpy()


def test_engine_matches_torch(setup):
    model, item_ids, pop, out = setup
    eng = Recommender(out)
    rng = np.random.default_rng(1)
    for alpha in (0.0, 0.75, 1.5):
        for _ in range(15):
            hist = rng.choice(N, size=int(rng.integers(1, 12)), replace=False)
            ref_items, ref_scores = torch_top(model, item_ids, pop, hist, 10, alpha, eng.scale)
            res = eng.recommend([int(item_ids[i]) for i in hist], 10, alpha)
            assert [r["item_id"] for r in res["items"]] == ref_items
            assert np.allclose([r["score"] for r in res["items"]], ref_scores, atol=1e-4)


def test_history_excluded_and_unknown_reported(setup):
    _, item_ids, _, out = setup
    eng = Recommender(out)
    hist = [int(item_ids[0]), int(item_ids[1]), 424242]
    res = eng.recommend(hist, 15)
    got = {r["item_id"] for r in res["items"]}
    assert int(item_ids[0]) not in got and int(item_ids[1]) not in got
    assert res["unknown_items"] == [424242]
    assert len(res["items"]) == 15 and not res["cold_start"]


def test_cold_start_returns_most_popular(setup):
    _, item_ids, pop, out = setup
    eng = Recommender(out)
    res = eng.recommend([], 5)
    assert res["cold_start"] and len(res["items"]) == 5
    assert res["items"][0]["item_id"] == int(item_ids[int(np.argmax(pop))])
    assert eng.recommend([999999], 3)["cold_start"]


def test_api(setup):
    _, item_ids, _, out = setup
    with TestClient(create_app(out)) as c:
        h = c.get("/health").json()
        assert h["status"] == "ok" and h["items"] == N
        r = c.post("/recommend", json={"history": [int(item_ids[3]), int(item_ids[4])], "k": 7})
        assert r.status_code == 200 and len(r.json()["items"]) == 7
        assert c.post("/recommend", json={"history": [int(item_ids[3])], "k": 0}).status_code == 422
        assert c.post("/recommend", json={"history": [], "alpha": 5}).status_code == 422
        assert c.get(f"/items/{int(item_ids[0])}").json()["title"].startswith("video")
        assert c.get("/items/1").status_code == 404
        a0 = c.post("/recommend", json={"history": [int(item_ids[3])], "k": 20, "alpha": 0.0}).json()["items"]
        a2 = c.post("/recommend", json={"history": [int(item_ids[3])], "k": 20, "alpha": 2.0}).json()["items"]
        assert [x["item_id"] for x in a0] != [x["item_id"] for x in a2]
EOF
find $ROOT -type f \( -name '*.py' -o -name '*.toml' -o -name Dockerfile \) | sort

/kaggle/working/vidreclens/Dockerfile
/kaggle/working/vidreclens/pyproject.toml
/kaggle/working/vidreclens/scripts/day1b_item_stats.py
/kaggle/working/vidreclens/scripts/day1_prepare.py
/kaggle/working/vidreclens/scripts/day2_join_features.py
/kaggle/working/vidreclens/scripts/day3_baselines.py
/kaggle/working/vidreclens/scripts/day4_twotower.py
/kaggle/working/vidreclens/scripts/day5_bias.py
/kaggle/working/vidreclens/scripts/day6_ab.py
/kaggle/working/vidreclens/scripts/day7_clusters.py
/kaggle/working/vidreclens/scripts/export_bundle.py
/kaggle/working/vidreclens/src/vidreclens/config.py
/kaggle/working/vidreclens/src/vidreclens/data/clean.py
/kaggle/working/vidreclens/src/vidreclens/data/eda.py
/kaggle/working/vidreclens/src/vidreclens/data/history.py
/kaggle/working/vidreclens/src/vidreclens/data/__init__.py
/kaggle/working/vidreclens/src/vidreclens/data/load.py
/kaggle/working/vidreclens/src/vidreclens/data/split.py
/kaggle/working/vidreclens/src/vidreclens/eval/__init__.py
/kagg

## 4. Install, fetch interaction files, clean data, join features

In [8]:
%%bash
cd /kaggle/working/vidreclens
pip install -q -e .
B=https://recsys.westlake.edu.cn/MicroLens-100k-Dataset
wget -q -c -P data/raw $B/MicroLens-100k_pairs.csv $B/MicroLens-100k_likes_and_views.txt
python scripts/day1_prepare.py
python scripts/day1b_item_stats.py
python scripts/day2_join_features.py --emb /kaggle/working/emb

{
  "raw_rows": 719405,
  "users": 98129,
  "items": 17228,
  "interactions": 705174,
  "sparsity": 0.9995828770789514,
  "seq_len_p50": 6.0,
  "seq_len_p90": 11.0,
  "seq_len_max": 203,
  "item_gini": 0.489552121098759,
  "top20pct_items_share": 0.5287276615416904,
  "span_days": 921.3602199074074
}
shape: (9, 7)
┌────────────┬─────────────┬─────────────┬───────────────┬──────────┬───────────┬───────────┐
│ statistic  ┆ iid         ┆ item        ┆ likes         ┆ views    ┆ like_rate ┆ log_views │
│ ---        ┆ ---         ┆ ---         ┆ ---           ┆ ---      ┆ ---       ┆ ---       │
│ str        ┆ f64         ┆ f64         ┆ f64           ┆ f64      ┆ f64       ┆ f64       │
╞════════════╪═════════════╪═════════════╪═══════════════╪══════════╪═══════════╪═══════════╡
│ count      ┆ 17228.0     ┆ 17228.0     ┆ 17228.0       ┆ 17228.0  ┆ 17228.0   ┆ 17228.0   │
│ null_count ┆ 0.0         ┆ 0.0         ┆ 0.0           ┆ 0.0      ┆ 0.0       ┆ 0.0       │
│ mean       ┆ 8613.5     

## 5. Baselines (popularity, item-kNN, content profiles)

In [9]:
if RUN_BASELINES:
    get_ipython().system("cd /kaggle/working/vidreclens && python scripts/day3_baselines.py")
else:
    print("skipped (RUN_BASELINES = False)")

skipped (RUN_BASELINES = False)


## 6. Two-tower models: ID only, content only, hybrid 3 modalities, hybrid all 4

In [10]:
if RUN_TWOTOWER:
    get_ipython().system("cd /kaggle/working/vidreclens && python scripts/day4_twotower.py")
else:
    print("skipped (RUN_TWOTOWER = False)")

skipped (RUN_TWOTOWER = False)


## 7. Refit on train+val for a fair test comparison with item-kNN

In [11]:
if RUN_REFIT:
    get_ipython().system("cd /kaggle/working/vidreclens && python scripts/day4_twotower.py --refit hybrid_all4=22")
else:
    print("skipped (RUN_REFIT = False)")

skipped (RUN_REFIT = False)


## 8. Popularity-bias mitigation (logit adjustment) with exposure metrics

In [12]:
if RUN_BIAS:
    get_ipython().system("cd /kaggle/working/vidreclens && python scripts/day5_bias.py")
else:
    print("skipped (RUN_BIAS = False)")

hybrid_all4 refit ep5 loss 7.8645 (111s)
hybrid_all4 refit ep10 loss 7.6542 (224s)
hybrid_all4 refit ep15 loss 7.5451 (337s)
hybrid_all4 refit ep20 loss 7.4674 (450s)
hybrid_all4 refit ep22 loss 7.4423 (495s)
alpha 0.0: recall@10 0.0684 tail 0.0396 coverage 0.695
alpha 0.25: recall@10 0.0693 tail 0.0444 coverage 0.793
alpha 0.5: recall@10 0.0706 tail 0.0491 coverage 0.877
alpha 0.75: recall@10 0.0691 tail 0.0534 coverage 0.919
alpha 1.0: recall@10 0.0654 tail 0.0567 coverage 0.924
hybrid_all4 refit ep5 loss 7.8801 (135s)
hybrid_all4 refit ep10 loss 7.6879 (270s)
hybrid_all4 refit ep15 loss 7.5895 (404s)
hybrid_all4 refit ep20 loss 7.5182 (539s)
hybrid_all4 refit ep22 loss 7.4942 (593s)
alpha 0.0: recall@10 0.0551 tail 0.0321 coverage 0.789
alpha 0.25: recall@10 0.0568 tail 0.0367 coverage 0.880
alpha 0.5: recall@10 0.0580 tail 0.0414 coverage 0.942
alpha 0.75: recall@10 0.0581 tail 0.0460 coverage 0.966
alpha 1.0: recall@10 0.0565 tail 0.0503 coverage 0.964

== val (train-only model, f

## 9. Evaluation framework: confidence intervals, paired tests, simulated A/B test, power analysis, segment slices

In [13]:
if RUN_AB:
    get_ipython().system("cd /kaggle/working/vidreclens && python scripts/day6_ab.py")
else:
    print("skipped (RUN_AB = False)")

loaded /kaggle/working/vidreclens/reports/twotower_hybrid_all4_refit.pt
scored itemknn
scored tt_a0
scored tt_a0.75

== 1. Policy metrics on 98129 test users (95% CI) ==
itemknn    hit@10 0.0586 [0.0571, 0.0600]   ndcg@10 0.0305 [0.0297, 0.0314]
tt_a0      hit@10 0.0551 [0.0537, 0.0565]   ndcg@10 0.0268 [0.0260, 0.0276]
tt_a0.75   hit@10 0.0581 [0.0566, 0.0595]   ndcg@10 0.0283 [0.0275, 0.0291]

== 2. Paired comparison, same users under both policies (B minus A) ==
A=tt_a0    B=tt_a0.75 hit@10 +0.0029 [+0.0019, +0.0039] p=0.0000   ndcg@10 +0.0015 [+0.0010, +0.0020] p=0.0000   hits only A/only B: 1109/1398
A=itemknn  B=tt_a0.75 hit@10 -0.0005 [-0.0019, +0.0009] p=0.4829   ndcg@10 -0.0022 [-0.0030, -0.0015] p=0.0000   hits only A/only B: 2463/2414
A=itemknn  B=tt_a0    hit@10 -0.0034 [-0.0048, -0.0021] p=0.0000   ndcg@10 -0.0037 [-0.0045, -0.0029] p=0.0000   hits only A/only B: 2468/2130

== 3. Simulated A/B test: users split at random, each sees only its arm's policy (1000 simulations p

## 10. Video clusters, per-cluster KPIs and an LLM-written summary (set LLM_MODEL = "none" to skip the LLM)

In [14]:
if RUN_CLUSTERS:
    get_ipython().system(f"cd /kaggle/working/vidreclens && python scripts/day7_clusters.py --llm {LLM_MODEL}")
else:
    print("skipped (RUN_CLUSTERS = False)")

loaded /kaggle/working/vidreclens/reports/twotower_hybrid_all4_refit.pt
scored itemknn
scored tt_a0
scored adj

== Clusters: composition and demand (k=12, KMeans on PCA of cover + title + CLIP RN50 embeddings) ==
cluster   items  catalog%  demand%  tail items%  log views  like rate   example titles
C0         1739      10.1     10.2         81.8      13.65      0.055   the inventory of those mist… | # Simpsons # Demon Son Bart…
C1         1221       7.1      5.9         85.3      14.14      0.040   three people A Bing respons… | only roadside stalls in Xin…
C2         1259       7.3      7.2         80.7      13.29      0.074   episode 3 | Hehuan vine ter… | # Comicrecommendations # Co…
C3         1369       7.9     14.7         58.0      13.94      0.047   is this your impression of … | you'll meet better that's w…
C4         1294       7.5      7.5         80.6      13.54      0.051   turtle was conferred | [God mosquito evolutionary …
C5         1381       8.0      5.0         89.0 

## 11. Export the serving bundle (small numpy-only model for the API) and run the tests

In [15]:
if RUN_EXPORT:
    get_ipython().system("cd /kaggle/working/vidreclens && python scripts/export_bundle.py")
    get_ipython().system("cd /kaggle/working/vidreclens && pip install -q pytest httpx fastapi && python -m pytest -q tests")
else:
    print("skipped (RUN_EXPORT = False)")

parity check on 200 users: identical top-10 for 200/200, largest score difference 3.81e-06
bundle /kaggle/working/serving_bundle (9.7 MB uncompressed) -> /kaggle/working/serving_bundle.zip
.........                                                                [100%]
=============================== warnings summary ===============================
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64
  /usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.p

## 12. Package the code and result files (no raw data, no checkpoints)

In [17]:
%%bash
cd /kaggle/working
rm -f vidreclens_repo.zip
zip -qr vidreclens_repo.zip vidreclens -x "vidreclens/data/*" "*.pt" "*/__pycache__/*" "*.egg-info/*"
ls -la vidreclens_repo.zip

-rw-r--r-- 1 root root 172496 Oct 25 13:30 vidreclens_repo.zip
